In [1]:
from flask import Flask, render_template, request, redirect, send_file
from extractors.indeed import extract_indeed_jobs
from extractors.wwr import extract_wwr_jobs
from file import save_to_file

app = Flask("JobScrapper")
db = {}

@app.route("/")
def home():
    return render_template("home.html", name="jo")

@app.route("/search")
def search():
    keyword = request.args.get("keyword")

    if keyword == None:
        return redirect("/")
    if keyword in db:
        jobs = db[keyword]
    else:
        indeed = extract_indeed_jobs(keyword)
        wwr = extract_wwr_jobs(keyword)
        jobs = indeed + wwr
        db[keyword] = jobs
        
    return render_template("search.html", keyword = keyword, jobs = jobs)

@app.route("/export")
def export():
    keyword = request.args.get("keyword")

    if keyword == None:
        return redirect("/")
    if keyword not in db:
        return redirect(f"/search?keyword={keyword}")    

    save_to_file(keyword, db[keyword])

    return send_file(f"{keyword}.csv", as_attachment=True)

app.run("0.0.0.0")

 * Serving Flask app 'JobScrapper'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.0.45:5000
Press CTRL+C to quit
192.168.0.45 - - [25/Nov/2024 19:45:14] "GET / HTTP/1.1" 200 -
